# Reproduce published methods (Phúc) - Progress Review 2
Việc: (1) RGT chính thức, (2) BotHP (`bothp.ipynb`), (3) SHGN nếu BotHP hỏng, (4) phương pháp tốt nhất trên `split()` của nhóm.

**Quy tắc:** mục 4-5 dùng split ngẫu nhiên của paper (lẫn test của nhóm) -> chỉ để kiểm tra số đã công bố, không tune mô hình nhóm. Mục 6 chỉ báo **validation**, test khóa.

**Trước khi chạy:** Runtime > Change runtime type > GPU. RGT nặng VRAM: T4 (15 GB) có thể OOM, ưu tiên **L4/A100** nếu có. Đổi runtime sẽ reset `/content`, chạy lại từ mục 0.

## 0. Kiểm tra runtime

In [ ]:
import sys, subprocess, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    print("!! Chưa bật GPU: Runtime > Change runtime type, rồi chạy lại từ đầu.")

## 1. Repo của nhóm + môi trường
Cài `requirements.txt` **trước**, rồi mới `install_pyg.py`.

In [ ]:
%cd /content
!test -d sna-content-moderation || git clone https://github.com/ThSonDev/sna-content-moderation.git
%cd /content/sna-content-moderation
!git pull -q
!pip install -q gdown
!pip install -q -r requirements.txt
!python install_pyg.py

## 2. Dữ liệu của nhóm (`data/MGTAB/*.pt`)

In [ ]:
import os, glob, shutil, subprocess, zipfile
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/MGTAB/features.pt"):
    subprocess.run(["gdown", "1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe", "-O", "MGTAB.zip"], check=True)
    zipfile.ZipFile("MGTAB.zip").extractall("data/_tmp")
    hit = glob.glob("data/_tmp/**/features.pt", recursive=True)
    assert hit, "Không thấy features.pt trong zip"
    shutil.move(os.path.dirname(hit[0]), "data/MGTAB")
    shutil.rmtree("data/_tmp", ignore_errors=True)
print(sorted(os.listdir("data/MGTAB")))

In [ ]:
# Self-check của nhóm: phải in "ok {...}". Đừng bấm dừng. Nếu treo: Runtime > Restart session, chạy lại từ mục 1.
!python mgtab.py

## 3. Bộ ghi kết quả

In [ ]:
import numpy as np, pandas as pd
RESULTS = []

def log_result(method, setting, acc=None, macro_f1=None, binary_f1=None, pr_auc=None, n_runs=None, note=""):
    RESULTS.append(dict(method=method, setting=setting, accuracy=acc, macro_f1=macro_f1,
                        binary_f1=binary_f1, pr_auc=pr_auc, n_runs=n_runs, note=note))

def save_results(path="/content/sna-content-moderation/results/reproduce.csv"):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df = pd.DataFrame(RESULTS).round(4)
    df.to_csv(path, index=False, encoding="utf-8-sig")   # utf-8-sig để Excel hiện đúng tiếng Việt
    return df

## 4. RGT chính thức (`RGT.py`, repo GraphDetec/MGTAB)
Mục tiêu paper: **acc 92.1 / macro-F1 90.4** (random 7:2:1, 5 seed, follower + friend). Mặc định của `RGT.py` đã đúng cấu hình paper (seed 1-5, hidden 128, dropout 0.5, weight decay 5e-3, quan hệ `0 1`).

Lưu ý: `MGTAB-GNN.py` **không** có RGT (chỉ GCN, GAT, GraphSage, RGCN, BotRGCN); RGT nằm ở `RGT.py`.

In [ ]:
%cd /content
!test -d MGTAB || git clone https://github.com/GraphDetec/MGTAB.git
%cd /content/MGTAB
!ls

In [ ]:
import os, shutil
os.chdir("/content/MGTAB")
print(os.popen("ls -la Dataset 2>&1; file Dataset/MGTAB 2>&1").read())

def remove_any(p):                      # xóa được cả file, symlink lẫn thư mục
    if os.path.islink(p) or os.path.isfile(p):
        os.remove(p)
    elif os.path.isdir(p):
        shutil.rmtree(p)

remove_any("Dataset/MGTAB")
if os.path.isfile("Dataset"):           # phòng khi chính "Dataset" là file
    os.remove("Dataset")
os.makedirs("Dataset", exist_ok=True)
shutil.copytree("/content/sna-content-moderation/data/MGTAB", "Dataset/MGTAB")
print(sorted(os.listdir("Dataset/MGTAB")))

In [ ]:
# Patch môi trường cho Colab (không đụng tới mô hình). Ghi vào cột "khác biệt" khi báo cáo:
#  1) CUDA_VISIBLE_DEVICES "8" -> "0" (Colab chỉ có GPU 0)
#  2) acc_xxx.item() -> float(acc_xxx) (sklearn mới trả float thuần, không có .item())
%cd /content/MGTAB
!sed -i 's/CUDA_VISIBLE_DEVICES"\] = "8"/CUDA_VISIBLE_DEVICES"] = "0"/' RGT.py
!sed -i -E 's/(acc_[a-z]+)\.item\(\)/float(\1)/g' RGT.py
!grep -n "CUDA_VISIBLE\|acc_train: \|acc_val: " RGT.py
# Kiểm tra loại F1 mà script tính (macro mới khớp 90.4 của paper):
!grep -n "f1_score\|average" RGT.py

In [ ]:
%cd /content/MGTAB
!grep -q "@torch.no_grad()" RGT.py || sed -i 's/^    def test():/    @torch.no_grad()\n    def test():/' RGT.py
!grep -n "no_grad" RGT.py
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python RGT.py --random_seed 1 2>&1 | grep -E "Epoch|OutOfMemory" | tail -5

In [ ]:
# Chạy đủ 5 seed theo mặc định paper (seed 1-5). Log lưu ở rgt_run.log; ẩn các dòng Epoch cho gọn.
%cd /content/MGTAB
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python RGT.py 2>&1 | tee rgt_run.log | grep -v "Epoch" | tail -40

In [ ]:
# DỰ PHÒNG nếu GPU nhỏ vẫn OOM: thu nhỏ mô hình. KHÔNG còn là tái lập đúng paper -> ghi rõ và báo nhóm.
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python RGT.py --hidden_dimension 64 --out_channel 32 2>&1 | tee rgt_run_small.log | grep -v "Epoch" | tail -40

In [ ]:
# Số từng seed (seed 1-5) lấy từ output của RGT.py ở cell trên. Paper: acc 92.1 / macro-F1 90.4.
rgt_acc      = [89.22, 90.39, 90.29, 90.29, 89.51]
rgt_macro_f1 = [85.43, 87.99, 87.86, 87.66, 86.76]
log_result("RGT (official)", "paper split, random 7:2:1, follow+friend",
           acc=np.mean(rgt_acc)/100, macro_f1=np.mean(rgt_macro_f1)/100, n_runs=len(rgt_acc),
           note="paper: 92.1/90.4; thấp hơn ~2.2/3.3 điểm; chưa rõ nguyên nhân (có thể paper dùng GNN_sample_large.py); khác biệt: dữ liệu nhóm, CUDA_VISIBLE_DEVICES 8->0, .item()->float(), @torch.no_grad() cho test(), torch 2.11/py3.13")

## 5. BotHP và SHGN
- **BotHP:** chạy trong `bothp.ipynb` (Colab riêng; cần Python 3.10 + torch 1.13). Repo chỉ có một lệnh cho MGTAB (`main_transductive.py --dataset MGTAB-FF`), không có cờ chọn biến thể RGCN/RGT.
- **F1 của BotHP là binary F1 của lớp bot** (`node_classification_evaluation.py` dòng 31 gọi `f1_score` không đặt `average`), nên không so trực tiếp với macro-F1 của RGT.
- **SHGN:** chỉ dùng nếu BotHP không cài được sau khi thử nghiêm túc (báo Sơn).

In [ ]:
# BotHP chạy ở bothp.ipynb (Colab riêng, 5 seed). Số lấy từ output của main_transductive.py:
# final_accuracy 0.9101±0.0014 | final_precision 0.8514 | final_recall 0.8048 | final_f1 0.8273±0.0046
# early-stopping_accuracy 0.9102 | early-stopping_f1 0.8289±0.0035
log_result("BotHP (MGTAB-FF)", "process.py split 70/20/10 seed 0, follow+friend, 5 seeds",
           acc=0.9101, binary_f1=0.8273, n_runs=5,
           note="final: acc 0.9101±0.0014, P 0.8514, R 0.8048, F1 0.8273±0.0046 (binary F1 lớp bot: node_classification_evaluation.py dòng 31 gọi f1_score không đặt average, nên mặc định binary; không so trực tiếp với macro-F1 của RGT); early-stopping: acc 0.9102, F1 0.8289; macro-F1 ước lượng ~0.883 từ P/R/acc, chưa tính từ dự đoán; Python 3.10 venv+uv, faiss-cpu, dữ liệu nhóm (labels_bot.pt->label.pt); split khác split RGT và split nhóm")

In [ ]:
# Dự phòng SHGN (bỏ comment nếu cần; xem argparse của SHGN.py trước):
# %cd /content/MGTAB
# !sed -n 1,40p SHGN.py
# !TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python SHGN.py 2>&1 | grep -v "Epoch" | tail -40

## 6. Phương pháp tốt nhất trên split của nhóm
Chọn theo macro-F1 cùng loại F1 từ mục 4-5. Dùng `split()` + 6 quan hệ từ `relations()`, báo `evaluate()` trên **validation**. Test khóa.

`SimpleRGCN` chỉ để **chạy thử pipeline** (split, edge, threshold, evaluate). **Không phải** kết quả tái lập, không đưa vào báo cáo. Khi có model chính thức của phương pháp thắng, thay vào `build()`.

In [ ]:
%cd /content/sna-content-moderation
import torch, torch.nn.functional as F
from sklearn.metrics import average_precision_score
from mgtab import load, relations, split, best_threshold, evaluate, RELATIONS

X, y, _, E, T, _ = load()
n = len(y)
rel = relations(E, T, n)
train, val, test = split(y)                      # test khóa: không dùng ở đây

src, dst, typ = [], [], []                       # 6 loại: follow=0 ... hashtag=5
for k, r in enumerate(RELATIONS):
    s, d = rel[r]
    if r in ("url", "hashtag"):                  # lưu một chiều -> thêm chiều ngược
        s, d = np.r_[s, d], np.r_[d, s]
    src.append(s); dst.append(d); typ.append(np.full(len(s), k))

dev = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.tensor(X, dtype=torch.float32, device=dev)
yt = torch.tensor(y, dtype=torch.long, device=dev)
edge_index = torch.tensor(np.vstack([np.concatenate(src), np.concatenate(dst)]), dtype=torch.long, device=dev)
edge_type = torch.tensor(np.concatenate(typ), dtype=torch.long, device=dev)
tr_t = torch.tensor(train, device=dev)
print(dev, x.shape, edge_index.shape, {r: len(rel[r][0]) for r in RELATIONS})

In [ ]:
from torch_geometric.nn import RGCNConv

class SimpleRGCN(torch.nn.Module):               # CHỈ smoke-test pipeline
    def __init__(self, in_dim, hid=128, n_rel=6, drop=0.5):
        super().__init__()
        self.lin = torch.nn.Linear(in_dim, hid)
        self.c1 = RGCNConv(hid, hid, n_rel); self.c2 = RGCNConv(hid, 2, n_rel); self.drop = drop
    def forward(self, x, ei, et):
        x = F.dropout(F.relu(self.lin(x)), self.drop, self.training)
        x = F.dropout(F.relu(self.c1(x, ei, et)), self.drop, self.training)
        return self.c2(x, ei, et)

def build(seed):
    torch.manual_seed(seed)
    # TODO: thay bằng model chính thức của phương pháp thắng, dùng hyperparameter của repo đó
    model = SimpleRGCN(x.shape[1]).to(dev)
    return model, torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

def run(model, opt, epochs=200, patience=30):
    best, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        F.cross_entropy(model(x, edge_index, edge_type)[tr_t], yt[tr_t]).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(model(x, edge_index, edge_type), 1)[:, 1].cpu().numpy()
        ap = average_precision_score(y[val], p[val])        # early stop theo PR-AUC validation
        if ap > best: best, bad, best_state = ap, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        return F.softmax(model(x, edge_index, edge_type), 1)[:, 1].cpu().numpy()

METHOD = "SimpleRGCN (smoke test)"               # đổi khi dùng model chính thức
rows = []
for seed in [0, 1, 2]:
    p = run(*build(seed))
    thr = best_threshold(y[val], p[val])                    # ngưỡng chọn trên validation như pipeline nhóm
    rows.append(evaluate(y[val], p[val], thr))
m = pd.DataFrame(rows).mean(numeric_only=True); s = pd.DataFrame(rows).std(numeric_only=True)
print(pd.DataFrame({"mean": m, "std": s}).round(4))

In [ ]:
is_smoke = "smoke" in METHOD
if not is_smoke:     # smoke test chỉ kiểm tra pipeline, không ghi vào CSV
    log_result(f"{METHOD} on group split", "split(), validation, one split, 3 init seeds", acc=m["accuracy"],
               macro_f1=m["macro_f1"], pr_auc=m["pr_auc"], n_runs=3,
               note="validation only; test locked; ngưỡng tune trên chính validation nên macro-F1/acc hơi lạc quan")

## 7. Lưu kết quả -> `results/reproduce.csv`
Tải file về hoặc copy lên Drive, vì Colab xóa hết khi ngắt.

In [ ]:
df = save_results()
df